In [ ]:
import torch 
import torch_geometric

In [98]:
X = torch.randn(64, 6)
X_sample = torch.randn(5,6)
# print(X)

labels = torch.tensor([0, 0, 0, 1, 1])
criterion = torch.nn.CrossEntropyLoss()


coords = torch.tensor([
    [0.0,  0.7,  0.0],   # Fz
    [0.3,  0.5,  0.0],   # F3
   [-0.3,  0.5,  0.0],   # F4
    [0.0,  0.0,  0.9],   # Cz
    [0.5,  0.0,  0.5],   # C4
])

node_names = ["Fz", "F3", "F4", "CZ", "C4"]

W = torch.randn(6,4)

A = torch.zeros(5,5)

def check_neighbors():
    for electrode in range(len(coords)): # Running over each electrode's coordinates
        for neighbor in range(len(coords)):
            if neighbor != electrode:
                distance_from_neighbor = 0.0
                for coord in range(len(coords[electrode])): # Running over each coordinate of electrode and the comparison electrode
                    distance_from_neighbor += (coords[electrode][coord] - coords[neighbor][coord]) ** 2
                
                distance_from_neighbor = distance_from_neighbor ** 0.5

            else:
                continue

            # print(f"Distance between: {node_names[electrode]} and {node_names[neighbor]} is: {distance_from_neighbor}")

            if distance_from_neighbor < 0.61:
                print(f"{node_names[electrode]} and {node_names[neighbor]} are neighbours")
            
            # else:
            #     print(f"{node_names[electrode]} and {node_names[neighbor]} are not neighbours")
            #     print(distance_from_neighbor)

In [99]:
class GCNLayer(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.W = torch.nn.Parameter(torch.randn(in_features, out_features))
        self.in_features = in_features
        self.out_features = out_features
    
    def forward(self, X, A):

        degrees = A.sum(dim=1)
        D_inv = torch.diag(1.0/degrees)

        normalized_A = torch.matmul(D_inv, A)
        normalized_result = torch.matmul(normalized_A, X)

        transformed_results = torch.matmul(normalized_result, self.W)

        return transformed_results

In [100]:
#Creating adjacency matrix
diff = coords.unsqueeze(0) - coords.unsqueeze(1) # Converts tensors to single dimensional tensors
dist = (diff ** 2).sum(dim=-1) ** 0.5  
A = (dist < 0.61).float() # Condition for each value in the tensor. if true => 1., if not => 0.
A.fill_diagonal_(0) # Fills diagonals with zeros

# Fixing degree-scale problem
identity_matrix = torch.eye(5)
A_hat = A + identity_matrix

model = GCNLayer(6, 4)

optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

In [102]:
for epoch in range(100):
    optimizer.zero_grad()
    out = model(X_sample, A_hat)
    loss = criterion(out, labels)
    loss.backward()
    optimizer.step()

    # if epoch % 10 == 0:
    #     print(f"Epoch {epoch}, Loss: {loss.item()}")

In [ ]:
from torch_geometric.nn import MessagePassing
import torch.nn.functional as F

class GATLayer(MessagePassing):
    def __init__(self, in_features, out_features):
        super().__init__(aggr='add')
        self.W = torch.nn.Linear()